In [1]:
!pip install beautifulsoup4 requests pyperclip

In [2]:
import re
import json
from collections import defaultdict

import requests
from bs4 import BeautifulSoup

DEBUG = False

In [3]:


def parse_number(text: str) -> int:
    return int(re.sub(r"[^\d]", "", text))


def parse_date(text: str) -> str:
    """
    Convert strings like:
        2026/04/02 19:00
        2026.04.02
    into:
        2026-04-02
    """
    m = re.search(r"(\d{4})[./-](\d{1,2})[./-](\d{1,2})", text)
    if not m:
        return text.strip()

    y, mo, d = map(int, m.groups())
    return f"{y:04d}-{mo:02d}-{d:02d}"


def parse_title(title: str):
    """
    Expected format:
        xxx【yyy】zzz応援プロジェクト
    """
    title = title.strip()

    m = re.match(r"(.+?)【(.+?)】(.+?)応援プロジェクト", title)

    if not m:
        return {
            "group": None,
            "member": None,
            "type": None,
        }

    group, member, typ = m.groups()

    return {
        "group": group.strip(),
        "member": member.strip(),
        "type": typ.strip(),
    }


def scrape_campfire(url: str):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/136.0 Safari/537.36"
        )
    }

    r = requests.get(url, headers=headers)
    r.raise_for_status()
    r.encoding = "utf-8"

    soup = BeautifulSoup(r.text, "html.parser")

    # ------------------------------------------------------------
    # title
    # ------------------------------------------------------------
    title_el = soup.select_one(
        "div.project-hero "
        "span.title-name"
    )

    title = title_el.get_text(strip=True) # type: ignore

    # ------------------------------------------------------------
    # overview
    # ------------------------------------------------------------
    total_el = soup.select_one(
        "div.overview p.backer-amount"
    )

    rest_el = soup.select_one(
        "div.overview p.backer"
    )

    total = parse_number(total_el.get_text()) # type: ignore
    rest = parse_number(rest_el.get_text()) # type: ignore

    # ------------------------------------------------------------
    # dates
    # ------------------------------------------------------------
    strongs = soup.select(
        "div.overview-bottom-pc "
        "div.text-when-closed-wrap p strong"
    )

    if DEBUG:
        print(strongs)

    # order:
    # start, rest, total, end
    start = parse_date(strongs[0].get_text())
    end = parse_date(strongs[3].get_text())

    
    # ------------------------------------------------------------
    # rewards
    # ------------------------------------------------------------
    reward_items = soup.select(
        "section#reward-pc-section "
        "div.reward-container ul li.item"
    )
    
    price_map = defaultdict(int)
    
    for item in reward_items:
        price_el = item.select_one(
            "div.summary div.info-area "
            "div.header p.price"
        )
    
        patrons_el = item.select_one(
            "div.summary div.info-area "
            "div.patrons-ship p.patrons"
        )
    
        if not price_el or not patrons_el:
            continue
        
        price = parse_number(price_el.get_text())
        patrons = parse_number(patrons_el.get_text())
    
        # sum patrons for duplicated price
        price_map[price] += patrons
    
    # sort by price ascending
    price_patrons = [
        {
            "price": price,
            "patrons": patrons,
        }
        for price, patrons in sorted(price_map.items())
    ]

    # ------------------------------------------------------------
    # subject
    # ------------------------------------------------------------
    subject = parse_title(title)

    # ------------------------------------------------------------
    # event year
    # ------------------------------------------------------------
    event_year = None

    if start:
        event_year = int(start[:4])

    result = {
        "url": url,
        "title": title,
        "total": total,
        "rest": rest,
        "per": "終了",
        "pricePatrons": price_patrons,
        "subject": subject,
        "start": start,
        "end": end,
        "eventYear": event_year,
    }

    return result


In [10]:


# ============================================================
# example
# ============================================================

cfid = "951521"
url = "https://camp-fire.jp/projects/" + cfid + "/view"

data = scrape_campfire(url)
txtOut = json.dumps(data, ensure_ascii=False, indent=4)

print(txtOut)

import pyperclip
pyperclip.copy(txtOut)
print("Copied to clipboard!")

{
    "url": "https://camp-fire.jp/projects/951521/view",
    "title": "chuLa【茉野ゆに】生誕祭応援プロジェクト",
    "total": 2643624,
    "rest": 114,
    "per": "終了",
    "pricePatrons": [
        {
            "price": 3000,
            "patrons": 7
        },
        {
            "price": 6000,
            "patrons": 1
        },
        {
            "price": 10000,
            "patrons": 89
        },
        {
            "price": 50000,
            "patrons": 14
        },
        {
            "price": 100000,
            "patrons": 4
        },
        {
            "price": 300000,
            "patrons": 2
        }
    ],
    "subject": {
        "group": "chuLa",
        "member": "茉野ゆに",
        "type": "生誕祭"
    },
    "start": "2026-05-15",
    "end": "2026-06-13",
    "eventYear": 2026
}
Copied to clipboard!
